In [ ]:
# ============================================================
# Hybrid CNN–Transformer–Random Forest Framework for IoT DDoS Detection
# ============================================================

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE, ADASYN
from sklearn.model_selection import train_test_split
import copy

# ---------------- CONFIG ----------------
class Config:
    INPUT_FEATURES = 25
    NUM_CLASSES = 5
    BATCH_SIZE = 64
    LOCAL_EPOCHS = 5
    NUM_ROUNDS = 10
    NUM_CLIENTS = 5
    LR = 0.001
    MU = 0.01
config = Config()

# ---------------- DATASET ----------------
class IoTDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

def generate_data(n=5000, balance="smote"):
    np.random.seed(42)
    X, y = [], []
    for c in range(config.NUM_CLASSES):
        samples = n // config.NUM_CLASSES
        X_c = np.random.multivariate_normal(mean=np.ones(config.INPUT_FEATURES)*c*2,
                                            cov=np.eye(config.INPUT_FEATURES), size=samples)
        y_c = np.full(samples, c)
        X.append(X_c); y.append(y_c)
    X, y = np.vstack(X), np.concatenate(y)
    X = MinMaxScaler().fit_transform(X)
    if balance=="smote": X,y = SMOTE().fit_resample(X,y)
    elif balance=="adasyn": X,y = ADASYN().fit_resample(X,y)
    elif balance=="smote+adasyn": X,y = SMOTE().fit_resample(X,y); X,y = ADASYN().fit_resample(X,y)
    return X,y

def create_clients(X,y):
    clients=[]; splits=np.array_split(range(len(X)), config.NUM_CLIENTS)
    for idx in splits:
        X_c,y_c=X[idx],y[idx]
        X_train,X_val,y_train,y_val=train_test_split(X_c,y_c,test_size=0.2)
        train_loader=DataLoader(IoTDataset(X_train,y_train),batch_size=config.BATCH_SIZE,shuffle=True)
        val_loader=DataLoader(IoTDataset(X_val,y_val),batch_size=config.BATCH_SIZE)
        clients.append((train_loader,val_loader))
    return clients

# ---------------- MODEL ----------------
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Linear(config.INPUT_FEATURES,128), nn.ReLU(),
            nn.Linear(128,256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128), nn.ReLU()
        )
        encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(128, config.NUM_CLASSES)
    def forward(self,x):
        x=self.cnn(x)
        x=x.unsqueeze(0) # seq_len=1
        x=self.transformer(x)
        return self.fc(x.squeeze(0))

# ---------------- TRAINING ----------------
def train(strategy="fedavg"):
    X,y=generate_data(balance="smote+adasyn")
    clients=create_clients(X,y)
    global_model=HybridModel()
    history_acc,history_f1=[],[]
    for rnd in range(config.NUM_ROUNDS):
        local_weights,client_sizes,local_steps=[],[],[]
        global_weights=copy.deepcopy(global_model.state_dict())
        for train_loader,_ in clients:
            local_model=copy.deepcopy(global_model)
            opt=optim.Adam(local_model.parameters(),lr=config.LR)
            steps=0; local_model.train()
            for _ in range(config.LOCAL_EPOCHS):
                for xb,yb in train_loader:
                    opt.zero_grad(); out=local_model(xb)
                    loss=nn.CrossEntropyLoss()(out,yb)
                    if strategy=="fedprox":
                        prox=0
                        for p,g in zip(local_model.parameters(),global_model.parameters()):
                            prox+=((p-g)**2).sum()
                        loss+=(config.MU/2)*prox
                    loss.backward(); opt.step(); steps+=1
            local_weights.append(copy.deepcopy(local_model.state_dict()))
            client_sizes.append(len(train_loader.dataset)); local_steps.append(steps)
        new_weights=copy.deepcopy(local_weights[0])
        if strategy in ["fedavg","fedprox"]:
            total=sum(client_sizes)
            for k in new_weights:
                new_weights[k]=sum(local_weights[i][k]*(client_sizes[i]/total) for i in range(len(local_weights)))
        elif strategy=="fednova":
            for k in new_weights:
                new_weights[k]=global_weights[k]+sum((local_weights[i][k]-global_weights[k])/local_steps[i] for i in range(len(local_weights)))/len(local_weights)
        global_model.load_state_dict(new_weights)
        # ---- Evaluation ----
        y_true,y_pred=[],[]
        global_model.eval()
        with torch.no_grad():
            for _,val_loader in clients:
                for xb,yb in val_loader:
                    out=global_model(xb); _,pred=out.max(1)
                    y_true.extend(yb.numpy()); y_pred.extend(pred.numpy())
        acc=accuracy_score(y_true,y_pred)*100
        f1=f1_score(y_true,y_pred,average="weighted")*100
        history_acc.append(acc); history_f1.append(f1)
        print(f"{strategy.upper()} Round {rnd+1}: Acc={acc:.2f}%, F1={f1:.2f}%")
    return history_acc,history_f1

# ---------------- RUN & PLOT ----------------
acc_avg,f1_avg=train("fedavg")
acc_prox,f1_prox=train("fedprox")
acc_nova,f1_nova=train("fednova")

plt.figure(); plt.plot(acc_avg,label="FedAvg"); plt.plot(acc_prox,label="FedProx"); plt.plot(acc_nova,label="FedNova")
plt.title("Accuracy Comparison"); plt.legend(); plt.show()

plt.figure(); plt.plot(f1_avg,label="FedAvg"); plt.plot(f1_prox,label="FedProx"); plt.plot(f1_nova,label="FedNova")
plt.title("F1 Score Comparison"); plt.legend(); plt.show()
